# Manga / Webtoon → 대사 추출 + 번역 (Colab)

파이프라인: `이미지/PDF → Koharu layout/segmentation → 말풍선 grouping → OCR → 언어 확인 → 외국어만 한국어 번역 → JSONL/TXT`

- 한국어: **Pororo/BrainOCR PyTorch CUDA** (CUDA 불가 시 CPU fallback)
- 일본어: MangaOCR
- 중국어/영어: PaddleOCR
- 말풍선/텍스트 소속은 Koharu segmentation 정보를 우선 사용합니다.
- 번역은 Qwen 1.7B/4B를 4-bit로 사용합니다.


## 1. 패키지 설치

처음 한 번 실행합니다. 설치 후 런타임을 자동 재시작합니다.

Pororo는 `comic-translate`의 유지보수 중인 PyTorch 구현을 사용하며, Colab의 기존 PyTorch/CUDA 환경을 그대로 사용합니다.


In [ ]:
from pathlib import Path
import os
import signal
import subprocess
import sys

환경_설치_표시 = Path("/content/.manga2text_environment_ready_v3")

if not 환경_설치_표시.exists():
    print("[설치 시작]")
    print("한국어 OCR: Pororo/BrainOCR PyTorch CUDA")
    print("일본어 OCR: MangaOCR")
    print("중국어/영어 OCR: PaddleOCR (CPU)")
    print("Colab의 기존 PyTorch/CUDA 환경은 유지합니다.")

    삭제할_패키지 = [
        "paddlepaddle",
        "paddlepaddle-gpu",
        "paddleocr",
        "paddlex",
        "onnxruntime",
        "onnxruntime-gpu",
        "langchain",
        "langchain-community",
        "langchain-text-splitters",
    ]
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", *삭제할_패키지],
        check=False,
    )

    설치할_패키지 = [
        "paddlepaddle==3.2.2",
        "paddleocr==3.3.2",
        "paddlex==3.3.13",
        "rfdetr==1.7.0",
        "safetensors>=0.5",
        "huggingface_hub>=0.27",
        "manga-ocr>=0.1.11",
        "transformers>=4.51",
        "accelerate>=1.2",
        "bitsandbytes>=0.45",
        "lingua-language-detector>=2.0",
        "pymupdf>=1.24",
        "pillow>=11.3.0",
        "numpy",
        "tqdm",
        "matplotlib",
        # comic-translate / Pororo PyTorch dependencies
        "mahotas>=1.4.18",
        "shapely>=2.1.1",
        "pyclipper>=1.3.0.post6",
        "six>=1.16.0",
        "requests>=2.31.0",
        "certifi",
        "wget>=3.2",
        "pyside6-essentials>=6.8.0",
        "msgpack>=1.1.0",
        "jaconv>=0.3.4",
        "setuptools",
    ]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *설치할_패키지],
        check=True,
    )

    subprocess.run(
        [
            sys.executable, "-m", "pip", "uninstall", "-y",
            "langchain", "langchain-community", "langchain-text-splitters",
        ],
        check=False,
    )

    환경_설치_표시.write_text("ready", encoding="utf-8")
    print("[설치 완료] 런타임을 자동 재시작합니다.")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("[설치 확인] 필요한 패키지는 이미 설치되어 있습니다.")


## 2. 저장소 가져오기

`manga2text_tmp`와 한국어 Pororo OCR 구현을 제공하는 `comic-translate`를 가져옵니다.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

저장소_경로 = Path("/content/manga2text_tmp")
comic_translate_경로 = Path("/content/comic-translate")

for path in [저장소_경로, comic_translate_경로]:
    if path.exists():
        shutil.rmtree(path)

subprocess.run(
    [
        "git", "clone", "-q",
        "https://github.com/HisameOgasahara/manga2text_tmp.git",
        str(저장소_경로),
    ],
    check=True,
)

subprocess.run(
    [
        "git", "clone", "--depth", "1", "-q",
        "https://github.com/ogkalu2/comic-translate.git",
        str(comic_translate_경로),
    ],
    check=True,
)

os.environ["COMIC_TRANSLATE_PATH"] = str(comic_translate_경로)

sys.path.insert(0, str(저장소_경로))
sys.path.insert(0, str(comic_translate_경로))

print("[저장소 준비 완료]")
print("manga2text    :", 저장소_경로)
print("comic-translate:", comic_translate_경로)


## 3. 실행 환경 확인

한국어 Pororo는 PyTorch CUDA를 사용합니다. CUDA를 사용할 수 없을 때만 CPU로 fallback합니다.


In [ ]:
import platform
from importlib.metadata import PackageNotFoundError, version

import paddle
import torch

def 설치_버전(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "설치 안 됨"

print("[실행 환경]")
print("Python              :", platform.python_version())
print("PyTorch             :", torch.__version__)
print("PyTorch CUDA        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU                  :", torch.cuda.get_device_name(0))

print("Paddle               :", paddle.__version__)
print("PaddleOCR            :", 설치_버전("paddleocr"))


## 4. 사용자 설정

- `글자_읽는_방법=자동`
  - 한국어 → Pororo/BrainOCR **PyTorch CUDA**
  - 일본어 → MangaOCR
  - 중국어/영어 → PaddleOCR
- 한국어 Pororo는 기본적으로 GPU를 사용합니다.


In [ ]:
import os
import sys
from pathlib import Path

os.environ["DISABLE_MODEL_SOURCE_CHECK"] = "True"

from manga2text_pipeline import (
    auto_detect_source_language,
    build_language_detector,
    classify_inputs,
    collect_page_images,
    describe_input_mode,
    load_koharu_detector,
    load_translation_model,
    make_preview_images,
    process_pages,
    save_results,
)
from manga2text_pororo import (
    install_pororo_overrides,
    load_ocr_backend,
    resolve_ocr_configuration,
)

# package 내부 process_pages가 Pororo run_ocr를 사용하도록 교체합니다.
install_pororo_overrides()

# @title 사용자 설정
언어_자동_판별 = True  # @param {type:"boolean"}
원문_언어 = "한국어"  # @param ["한국어", "일본어", "중국어", "영어"]

글자_읽는_방법 = "자동"  # @param ["자동", "PororoOCR", "MangaOCR", "PaddleOCR"]
읽기_방향 = "자동"  # @param ["자동", "오른쪽→왼쪽 (일본 만화)", "왼쪽→오른쪽 (웹툰/영문)"]

외국어_한국어_번역 = True  # @param {type:"boolean"}
번역_모델 = "Qwen3-1.7B (가볍고 빠름)"  # @param ["Qwen3-1.7B (가볍고 빠름)", "Qwen3-4B (품질 우선)"]

효과음도_읽기 = False  # @param {type:"boolean"}
PDF_화질_DPI = 200  # @param {type:"integer"}
처리할_페이지_수 = 0  # @param {type:"integer"}
동시에_준비할_작업_수 = 4  # @param {type:"integer"}
진단_로그_보기 = True  # @param {type:"boolean"}

언어_코드 = {
    "한국어": "ko",
    "일본어": "ja",
    "중국어": "zh",
    "영어": "en",
}
OCR_코드 = {
    "자동": "auto",
    "PororoOCR": "pororo",
    "MangaOCR": "manga",
    "PaddleOCR": "paddle",
}
읽기_방향_코드 = {
    "자동": "auto",
    "오른쪽→왼쪽 (일본 만화)": "rtl",
    "왼쪽→오른쪽 (웹툰/영문)": "ltr",
}
번역_모델_코드 = {
    "Qwen3-1.7B (가볍고 빠름)": "Qwen/Qwen3-1.7B",
    "Qwen3-4B (품질 우선)": "Qwen/Qwen3-4B",
}

AUTO_DETECT_SOURCE_LANGUAGE = 언어_자동_판별
SOURCE_LANGUAGE = 언어_코드[원문_언어]
OCR_BACKEND = OCR_코드[글자_읽는_방법]
READING_DIRECTION = 읽기_방향_코드[읽기_방향]

ENABLE_TRANSLATION = 외국어_한국어_번역
TRANSLATION_MODEL = 번역_모델_코드[번역_모델]
INCLUDE_SFX = 효과음도_읽기

PDF_DPI = PDF_화질_DPI
PAGE_LIMIT = None if 처리할_페이지_수 <= 0 else 처리할_페이지_수
INPUT_WORKERS = max(1, 동시에_준비할_작업_수)
DEBUG_LOG = 진단_로그_보기

PADDLE_DEVICE = "cpu"
PORORO_DEVICE = "cuda"

MAX_NEW_TOKENS = 256
CROP_PADDING = 8
ROW_TOLERANCE = 80
AUTO_LANGUAGE_SAMPLE_CROPS = 3
DEBUG_SAMPLES_PER_PAGE = 3

CLASS_THRESHOLDS = {
    0: 0.25,
    1: 0.20,
    2: 0.50,
    3: 0.50,
}

WORK_DIR = Path("/content/manga2text")
INPUT_DIR = WORK_DIR / "input"
PAGE_DIR = WORK_DIR / "pages"
OUTPUT_DIR = WORK_DIR / "output"

for directory in [INPUT_DIR, PAGE_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("[현재 설정]")
print("언어 자동 판별 :", AUTO_DETECT_SOURCE_LANGUAGE)
print("글자 읽는 방법 :", 글자_읽는_방법)
print("Pororo 장치     :", PORORO_DEVICE)
print("외국어 번역     :", ENABLE_TRANSLATION)


## 4-A. 읽기 순서 설정 (선택)

필요할 때만 실행합니다. 프리셋 또는 수동 설정을 고를 수 있습니다.


In [ ]:
from reading_order_ui import show_reading_order_controls

reading_order_ui = show_reading_order_controls()


## 5. 만화 업로드 + 미리보기

In [ ]:
import shutil
import matplotlib.pyplot as plt
from google.colab import files

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded_files = files.upload()
for filename, file_bytes in uploaded_files.items():
    (INPUT_DIR / filename).write_bytes(file_bytes)

input_groups = classify_inputs(INPUT_DIR)
print("[입력 확인]")
print("입력 종류 :", describe_input_mode(input_groups))
print("이미지 수 :", len(input_groups["images"]))
print("PDF 수    :", len(input_groups["pdfs"]))

preview_items = make_preview_images(
    input_dir=INPUT_DIR,
    max_items=8,
    pdf_preview_pages=3,
)

if not preview_items:
    raise RuntimeError("미리보기 가능한 이미지 또는 PDF가 없습니다.")

cols = min(4, len(preview_items))
rows = (len(preview_items) + cols - 1) // cols
plt.figure(figsize=(4 * cols, 5 * rows))
for i, (label, image) in enumerate(preview_items, start=1):
    ax = plt.subplot(rows, cols, i)
    ax.imshow(image)
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 6. 페이지 준비 + 언어 자동 판별

In [ ]:
import shutil

if PAGE_DIR.exists():
    shutil.rmtree(PAGE_DIR)
PAGE_DIR.mkdir(parents=True, exist_ok=True)

page_paths = collect_page_images(
    input_dir=INPUT_DIR,
    page_dir=PAGE_DIR,
    pdf_dpi=PDF_DPI,
    page_limit=PAGE_LIMIT,
    workers=INPUT_WORKERS,
)

print("준비된 페이지:", len(page_paths))

detector = load_koharu_detector()

if AUTO_DETECT_SOURCE_LANGUAGE:
    selected_source_language, auto_language_details = auto_detect_source_language(
        page_paths=page_paths,
        detector=detector,
        class_thresholds=CLASS_THRESHOLDS,
        crop_padding=CROP_PADDING,
        max_crops=AUTO_LANGUAGE_SAMPLE_CROPS,
        paddle_device=PADDLE_DEVICE,
    )
else:
    selected_source_language = SOURCE_LANGUAGE

ocr_config = resolve_ocr_configuration(
    source_language=selected_source_language,
    ocr_backend=OCR_BACKEND,
    reading_direction=READING_DIRECTION,
)

print("[OCR routing]")
print(ocr_config)


## 7. OCR / 번역 모델 로드

한국어면 Pororo/BrainOCR **PyTorch CUDA**를 GPU에서 로드합니다.


In [ ]:
import torch

ocr_model = load_ocr_backend(
    backend=ocr_config["ocr_backend"],
    paddle_lang=ocr_config["paddle_lang"],
    paddle_device=PADDLE_DEVICE,
    pororo_device=PORORO_DEVICE,
)

language_detector, language_to_code = build_language_detector()

translation_tokenizer = None
translation_model = None

if ENABLE_TRANSLATION and selected_source_language != "ko":
    translation_tokenizer, translation_model = load_translation_model(
        TRANSLATION_MODEL
    )

print("[모델 준비 완료]")
print("OCR backend:", ocr_config["ocr_backend"])
if ocr_config["ocr_backend"] == "pororo":
    print("Pororo backend: PyTorch/BrainOCR")
    print("Pororo GPU 요청:", PORORO_DEVICE)


## 8. 전체 페이지 처리

Koharu가 `panel / bubble / text` 영역을 검출·그룹핑하고, 한국어 text crop의 인식은 Pororo/BrainOCR PyTorch가 담당합니다.


In [ ]:
records = process_pages(
    page_paths=page_paths,
    detector=detector,
    ocr_backend=ocr_config["ocr_backend"],
    ocr_model=ocr_model,
    language_detector=language_detector,
    language_to_code=language_to_code,
    class_thresholds=CLASS_THRESHOLDS,
    reading_direction=ocr_config["reading_direction"],
    row_tolerance=ROW_TOLERANCE,
    crop_padding=CROP_PADDING,
    include_sfx=INCLUDE_SFX,
    enable_translation=ENABLE_TRANSLATION,
    translation_tokenizer=translation_tokenizer,
    translation_model=translation_model,
    max_new_tokens=MAX_NEW_TOKENS,
    debug=DEBUG_LOG,
    debug_samples_per_page=DEBUG_SAMPLES_PER_PAGE,
)

print("records:", len(records))


## 9. 결과 저장 + 미리보기

In [ ]:
import shutil

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

jsonl_path, txt_path = save_results(
    records=records,
    output_dir=OUTPUT_DIR,
)

print()
for record in records[:50]:
    original = record["original"].replace("\n", " ")
    korean = record["korean"].replace("\n", " ")
    print(
        f"[p.{record['page']:03d} / {record['order']:02d}] "
        f"{record['language']} | {original}"
        + (f" -> {korean}" if korean != original else "")
    )


## 10. 결과 다운로드

In [ ]:
from google.colab import files

files.download(str(txt_path))
files.download(str(jsonl_path))
